In [ ]:
import os
import pickle
import pandas as pd

# Assume "df" is produced by:
# df = rec.geo3d.rename("value").to_dataframe()
# and it has a MultiIndex [label, digitized] with columns "type" and "value"

# Pivot the dataframe so that each label becomes a row and digitized values become x, y, z 
df_reset = df.reset_index()  # now columns: label, digitized, type, value
df_wide = df_reset.pivot(index="label", columns="digitized", values="value")
df_wide.columns = ["x", "y", "z"]

# Retrieve the 'type' column (same for each label) 
type_series = df_reset.groupby("label")["type"].first()

# Combine the type info with the pivoted DataFrame
df_wide["type"] = type_series

# Reset index and rename the label column to "name"
df_final = df_wide.reset_index().rename(columns={"label": "name"})
df_final = df_final[["name", "type", "x", "y", "z"]]

# Optionally, convert the type string, e.g. "PointType.SOURCE" -> "source"
df_final["type"] = df_final["type"].apply(lambda s: s.split('.')[-1].lower())

# Create the output directory (assuming root_dir\derivatives)
out_dir = os.path.join("root_dir", "derivatives")
os.makedirs(out_dir, exist_ok=True)

# Save the optodes table as a TSV file
tsv_path = os.path.join(out_dir, "optodes.tsv")
df_final.to_csv(tsv_path, sep="\t", index=False)

# Save geo3Dscan as a pickle file (choose any filename; e.g., geo3Dscan.pkl)
pkl_path = os.path.join(out_dir, "geo3Dscan.pkl")
with open(pkl_path, "wb") as f:
    pickle.dump(geo3Dscan, f)